# Random Forest robusto con optimización de hiperparámetros y métricas completas
Este notebook replica la estructura robusta de tus modelos previos, usando RandomForestClassifier, GridSearchCV y todas las métricas y visualizaciones.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize, StandardScaler
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [ ]:
# === CONFIGURACIÓN ===
INPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23'
BASE_OUTPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/models'
MODEL_NAME = 'RandomForest'
INTENTO = 4
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f'{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}')
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Selecciona aquí qué métodos de balance ejecutar: 'ALL' o lista p.e. ['SMOTE']
# Recomendado: usar una lista, p.e. ['NONE'] o ['SMOTE'] o 'ALL' para ejecutar todos
RUN_BALANCE_METHODS = ['ALL']  # opciones: 'ALL' o lista como ['NONE','SMOTE']

# Control de paralelismo del GridSearch (reduce memoria si pones 1)
GRID_N_JOBS = 1
# --- Preferencias de reducción y búsqueda ---
# Si tus datos ya vienen normalizados y consistentes, pon True para NO re-escalar dentro del notebook
USE_EXISTING_SCALING = True
# Activar modo reducido para ahorrar tiempo/memoria (usa RandomizedSearch, menos variantes y normalizaciones)
REDUCED_PLAN = True
# Si quieres usar RandomizedSearchCV en modo reducido, controla n_iter
N_ITER_RS = 20
# Límite de variantes por método de balance en REDUCED_PLAN
MAX_VARIANTS_PER_BALANCE = 5
# Normalizaciones a probar cuando REDUCED_PLAN está activo
NORMALIZATIONS_REDUCED = [None, 'standard']

# Feature selection opcional (usa un RF pequeño para seleccionar TOP_K features).
USE_FEATURE_SELECTION = False
TOP_K = 50

ALL_BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

# Definición de espacios de hiperparámetros (completo y reducido)
param_grid_full = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 10, 20, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt']
}

param_grid_reduced = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

# Por compatibilidad con código posterior, definir param_grid según modo
param_grid = param_grid_reduced if REDUCED_PLAN else param_grid_full

# Normalizar formatos de petición y construir BALANCE_METHODS robustamente
# Acepta: 'ALL' (string), ['ALL'] (lista), o lista con nombres exactos como ['SMOTE']
import json
def load_checkpoint(path):
    if os.path.exists(path):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print('No se pudo leer checkpoint (se ignorará):', e)
            return {}
    return {}

def save_checkpoint(path, data):
    try:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    except Exception as e:
        print('Error guardando checkpoint:', e)

# Función de ayuda: detectar si un dataset parece ya normalizado
def detect_scaling(df, n_cols=10):
    """Inspecciona una muestra de columnas y detecta si los datos parecen min-max o z-score normalizados.
    Devuelve: 'minmax', 'zscore' o 'none'.
    """
    try:
        sample = df.iloc[:, :n_cols]
    except Exception:
        sample = df
    stats = sample.describe()
    mins = stats.loc['min']
    maxs = stats.loc['max']
    means = stats.loc['mean']
    stds = stats.loc['std']

    # criterio min-max: la mayoría de features tienen min en [0-0.05] y max en [0.95-1.05]
    minmax_count = ((mins >= -0.05) & (maxs <= 1.05)).sum()
    # criterio z-score: media cerca de 0 y std cerca de 1
    zscore_count = ((means.abs() <= 0.2) & (stds.between(0.7, 1.3))).sum()
    total = len(mins)
    if total == 0:
        return 'none'
    if minmax_count / total >= 0.7:
        return 'minmax'
    if zscore_count / total >= 0.7:
        return 'zscore'
    return 'none'

# Resolver 'ALL' en cualquier forma y construir requested
requested = None
if isinstance(RUN_BALANCE_METHODS, str):
    if RUN_BALANCE_METHODS.upper() == 'ALL':
        BALANCE_METHODS = ALL_BALANCE_METHODS
    else:
        requested = [RUN_BALANCE_METHODS]
elif isinstance(RUN_BALANCE_METHODS, (list, tuple, set)):
    upper_vals = [str(x).upper() for x in RUN_BALANCE_METHODS]
    if 'ALL' in upper_vals:
        BALANCE_METHODS = ALL_BALANCE_METHODS
    else:
        requested = list(RUN_BALANCE_METHODS)
else:
    requested = [RUN_BALANCE_METHODS]

if requested is not None:
    BALANCE_METHODS = {k: v for k, v in ALL_BALANCE_METHODS.items() if k in requested}
    missing = set(requested) - set(BALANCE_METHODS.keys())
    if missing:
        print(f"Advertencia: estos métodos solicitados no existen y serán ignorados: {missing}")

# Rutas de checkpoint/resumen para reanudación incremental
CHECKPOINT_FILE = os.path.join(OUTPUT_PATH, 'checkpoint_rf.json')
RESUMEN_CSV = os.path.join(OUTPUT_PATH, 'resumen_modelos_randomforest.csv')
FAILED_CSV = os.path.join(OUTPUT_PATH, 'resumen_failed_randomforest.csv')

metricas_totales = []

variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])

In [ ]:
# === EJECUCIÓN: GridSearch / RandomizedSearch + guardado incremental + checkpoint ===
# REDUCED_PLAN y configuraciones ya definidas en la celda anterior
# Construir set de completados desde RESUMEN_CSV y checkpoint (si existen)
completed = set()
if os.path.exists(RESUMEN_CSV):
    try:
        df_done = pd.read_csv(RESUMEN_CSV)
        for _, row in df_done.iterrows():
            norm = row.get('normalization', 'none') if 'normalization' in row.index else 'none'
            completed.add((str(row.get('modelo')), str(row.get('balanceo')), str(norm)))
        print(f'Encontrados {len(completed)} experimentos completados en {RESUMEN_CSV}')
    except Exception as e:
        print('No se pudo leer RESUMEN_CSV:', e)

# cargar checkpoint y obtener completados (keys formato modelo|balanceo|normalization)
checkpoint = load_checkpoint(CHECKPOINT_FILE)
for key, info in checkpoint.items():
    try:
        parts = key.split('|')
        if len(parts) == 3:
            modelo, balanceo, norm = parts
        elif len(parts) == 2:
            modelo, balanceo = parts
            norm = 'none'
        else:
            continue
        if info.get('state') == 'completed':
            completed.add((modelo, balanceo, norm))
    except Exception:
        continue

import traceback

# Copiar y posiblemente filtrar balance methods en modo reducido
balances_dict = dict(BALANCE_METHODS)
if REDUCED_PLAN:
    # eliminar NONE por defecto para ahorrar tiempo (a menos que el usuario lo haya pedido explícitamente)
    if 'NONE' in balances_dict:
        balances_dict.pop('NONE')

for balance_name, balancer in balances_dict.items():
    print(f'=== Balanceo: {balance_name} ===')
    output_subdir = os.path.join(OUTPUT_PATH, balance_name)
    os.makedirs(output_subdir, exist_ok=True)

    variantes_all = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])
    # seleccionar subset de variantes si REDUCED_PLAN
    if REDUCED_PLAN and len(variantes_all) > MAX_VARIANTS_PER_BALANCE:
        idxs = np.unique(np.linspace(0, len(variantes_all) - 1, num=MAX_VARIANTS_PER_BALANCE, dtype=int))
        variantes = [variantes_all[i] for i in idxs]
    else:
        variantes = variantes_all

    # normalizaciones a probar
    normalizations = NORMALIZATIONS_REDUCED if REDUCED_PLAN else [None]

    for x_file in variantes:
        base_name = x_file.replace('X_train_', '').replace('.parquet', '')

        for norm in normalizations:
            norm_label = norm if norm is not None else 'none'
            key = f"{base_name}|{balance_name}|{norm_label}"
            # Saltar si ya completado según CSV o checkpoint
            if (base_name, balance_name, norm_label) in completed:
                print(f"Skipping {base_name} | {balance_name} | {norm_label} (already completed)")
                continue

            try:
                # marcar como running en checkpoint
                checkpoint.setdefault(key, {})
                checkpoint[key].update({'state': 'running', 'start': str(datetime.now())})
                save_checkpoint(CHECKPOINT_FILE, checkpoint)

                print(f'🔍 Procesando: {base_name} | {norm_label}')
                X_train = pd.read_parquet(os.path.join(INPUT_PATH, f'X_train_{base_name}.parquet'))
                X_test = pd.read_parquet(os.path.join(INPUT_PATH, f'X_test_{base_name}.parquet'))
                y_train = pd.read_parquet(os.path.join(INPUT_PATH, f'y_train_{base_name}.parquet')).squeeze()
                y_test = pd.read_parquet(os.path.join(INPUT_PATH, f'y_test_{base_name}.parquet')).squeeze()
                y_train = y_train - y_train.min()
                y_test = y_test - y_test.min()

                # detectar si los datos parecen ya normalizados
                detected_scaling = detect_scaling(X_train)
                if USE_EXISTING_SCALING or detected_scaling in ('minmax', 'zscore'):
                    if detected_scaling in ('minmax', 'zscore'):
                        print(f"Nota: detectado formato {detected_scaling} en {base_name} — omitiendo re-escalado adicional.")
                    X_train_proc = X_train.copy()
                    X_test_proc = X_test.copy()
                else:
                    # aplicar normalización antes del resampling si se solicitó 'standard' y no hay scaling existente
                    if norm == 'standard':
                        scaler = StandardScaler()
                        X_train_proc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
                        X_test_proc = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
                    else:
                        X_train_proc = X_train.copy()
                        X_test_proc = X_test.copy()

                # balancear (resample) sobre X_train_proc
                if balancer is not None:
                    X_train_bal, y_train_bal = balancer.fit_resample(X_train_proc, y_train)
                else:
                    X_train_bal, y_train_bal = X_train_proc, y_train

                cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                rf = RandomForestClassifier(class_weight='balanced', random_state=42)

                # Seleccionar búsqueda: RandomizedSearchCV en modo reducido, GridSearch en modo completo
                if REDUCED_PLAN:
                    try:
                        from sklearn.model_selection import RandomizedSearchCV
                        search = RandomizedSearchCV(rf, param_distributions=param_grid, n_iter=N_ITER_RS, cv=cv, scoring='f1_macro', n_jobs=GRID_N_JOBS, random_state=42)
                    except Exception as e:
                        print('RandomizedSearchCV no disponible, usando GridSearchCV como fallback:', e)
                        search = GridSearchCV(rf, param_grid=param_grid, cv=cv, scoring='f1_macro', n_jobs=GRID_N_JOBS)
                else:
                    search = GridSearchCV(rf, param_grid=param_grid, cv=cv, scoring='f1_macro', n_jobs=GRID_N_JOBS)

                search.fit(X_train_bal, y_train_bal)

                best_rf = search.best_estimator_
                cv_f1_mean = getattr(search, 'best_score_', np.nan)
                # intentar obtener std_test_score si existe
                try:
                    best_idx = search.best_index_
                    cv_f1_std = search.cv_results_.get('std_test_score', [np.nan])[best_idx]
                except Exception:
                    cv_f1_std = np.nan

                start = time.time()
                best_rf.fit(X_train_bal, y_train_bal)
                tiempo = (time.time() - start) / 60

                y_pred_train = best_rf.predict(X_train_bal)
                # predecir sobre X_test_proc
                y_pred_test = best_rf.predict(X_test_proc)

                if hasattr(best_rf, 'predict_proba'):
                    y_proba = best_rf.predict_proba(X_test_proc)
                else:
                    y_proba = np.zeros((len(X_test_proc), len(np.unique(y_test))))

                report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True)).T
                report_train = pd.DataFrame(classification_report(y_train_bal, y_pred_train, output_dict=True)).T
                report_test['set'] = 'test'
                report_train['set'] = 'train'
                full_report = pd.concat([report_train, report_test])
                full_report['tiempo_min'] = tiempo
                full_report.to_csv(os.path.join(output_subdir, f'metricas_{base_name}_{norm_label}.csv'))

                cm = confusion_matrix(y_test, y_pred_test)
                with PdfPages(os.path.join(output_subdir, f'reporte_{base_name}_{norm_label}.pdf')) as pdf:
                    ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
                    plt.title('Matriz de Confusión')
                    pdf.savefig(); plt.close()

                    classes = np.unique(y_test)
                    y_bin = label_binarize(y_test, classes=classes)
                    plt.figure()
                    for i, clase in enumerate(classes):
                        try:
                            fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                            roc_auc = auc(fpr, tpr)
                            plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
                        except Exception:
                            continue
                    plt.plot([0, 1], [0, 1], 'k--')
                    plt.title('Curvas ROC por clase')
                    plt.xlabel('FPR')
                    plt.ylabel('TPR')
                    plt.legend()
                    pdf.savefig(); plt.close()

                    plt.figure()
                    for i, clase in enumerate(classes):
                        try:
                            precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                            pr_auc = average_precision_score(y_bin[:, i], y_proba[:, i])
                            plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
                        except Exception:
                            continue
                    plt.title('Curvas Precision-Recall por clase')
                    plt.xlabel('Recall')
                    plt.ylabel('Precision')
                    plt.legend()
                    pdf.savefig(); plt.close()

                resumen = {
                    'modelo': base_name,
                    'balanceo': balance_name,
                    'normalization': norm_label,
                    'accuracy_test': accuracy_score(y_test, y_pred_test),
                    'macro_f1_test': report_test.loc['macro avg', 'f1-score'],
                    'weighted_f1_test': report_test.loc['weighted avg', 'f1-score'],
                    'accuracy_train': accuracy_score(y_train_bal, y_pred_train),
                    'macro_f1_train': report_train.loc['macro avg', 'f1-score'],
                    'weighted_f1_train': report_train.loc['weighted avg', 'f1-score'],
                    'tiempo_min': tiempo,
                    'sobreajuste_macro_f1': report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
                    'cv_macro_f1_mean': cv_f1_mean,
                    'cv_macro_f1_std': cv_f1_std
                }
                for clase in report_test.index:
                    if clase not in ['accuracy', 'macro avg', 'weighted avg']:
                        resumen[f'f1_{clase}_test'] = report_test.loc[clase, 'f1-score']
                        resumen[f'recall_{clase}_test'] = report_test.loc[clase, 'recall']
                        resumen[f'precision_{clase}_test'] = report_test.loc[clase, 'precision']
                        resumen[f'f1_{clase}_train'] = report_train.loc[clase, 'f1-score']
                        resumen[f'recall_{clase}_train'] = report_train.loc[clase, 'recall']
                        resumen[f'precision_{clase}_train'] = report_train.loc[clase, 'precision']

                metricas_totales.append(resumen)
                # Guardado incremental: escribir la fila al CSV de resumen inmediatamente
                row_df = pd.DataFrame([resumen])
                if not os.path.exists(RESUMEN_CSV):
                    row_df.to_csv(RESUMEN_CSV, index=False)
                else:
                    row_df.to_csv(RESUMEN_CSV, index=False, header=False, mode='a')

                # Añadir al set de completados para que no se reprocese en esta ejecución
                completed.add((base_name, balance_name, norm_label))
                # marcar como completed en checkpoint
                checkpoint[key].update({'state': 'completed', 'end': str(datetime.now()), 'tiempo_min': tiempo})
                save_checkpoint(CHECKPOINT_FILE, checkpoint)

                print(f"✅ {base_name} [{balance_name}] [{norm_label}]: F1_clase_0_test={resumen.get('f1_0_test', 0):.3f}, Recall_clase_0_test={resumen.get('recall_0_test', 0):.3f}, Tiempo={tiempo:.2f}min")
            except Exception as e:
                tb = traceback.format_exc()
                print(f'❌ Error en {base_name} con balanceo {balance_name} y norm {norm_label}: {e}')
                # Registrar fallo en CSV para poder reintentar o depurar
                fail_row = pd.DataFrame([{'modelo': base_name, 'balanceo': balance_name, 'normalization': norm_label, 'error': str(e), 'traceback': tb, 'time': datetime.now()}])
                if not os.path.exists(FAILED_CSV):
                    fail_row.to_csv(FAILED_CSV, index=False)
                else:
                    fail_row.to_csv(FAILED_CSV, index=False, header=False, mode='a')
                # marcar como failed en checkpoint
                checkpoint.setdefault(key, {})
                checkpoint[key].update({'state': 'failed', 'error': str(e), 'traceback': tb, 'time': str(datetime.now())})
                save_checkpoint(CHECKPOINT_FILE, checkpoint)

# Además guardamos el dataframe final agregado (opcional) para compatibilidad con código previo
df_final = pd.DataFrame(metricas_totales)
if not df_final.empty:
    df_final.to_csv(os.path.join(OUTPUT_PATH, 'resumen_modelos_randomforest.csv'), index=False)
print(f"\n🏆 Resumen de todas las variantes guardado en: {os.path.join(OUTPUT_PATH, 'resumen_modelos_randomforest.csv')}")

No se pudo leer RESUMEN_CSV: No columns to parse from file
=== Balanceo: NONE ===
🔍 Procesando: MaxAbs_FE_ALL
✅ MaxAbs_FE_ALL [NONE]: F1_clase_0_test=0.124, Recall_clase_0_test=0.077, Tiempo=12.25min
🔍 Procesando: MaxAbs_FE_ANOVA
✅ MaxAbs_FE_ALL [NONE]: F1_clase_0_test=0.124, Recall_clase_0_test=0.077, Tiempo=12.25min
🔍 Procesando: MaxAbs_FE_ANOVA
✅ MaxAbs_FE_ANOVA [NONE]: F1_clase_0_test=0.230, Recall_clase_0_test=0.415, Tiempo=1.02min
🔍 Procesando: MaxAbs_FE_MI
✅ MaxAbs_FE_ANOVA [NONE]: F1_clase_0_test=0.230, Recall_clase_0_test=0.415, Tiempo=1.02min
🔍 Procesando: MaxAbs_FE_MI
✅ MaxAbs_FE_MI [NONE]: F1_clase_0_test=0.233, Recall_clase_0_test=0.327, Tiempo=2.41min
🔍 Procesando: MaxAbs_FE_PCA30
✅ MaxAbs_FE_MI [NONE]: F1_clase_0_test=0.233, Recall_clase_0_test=0.327, Tiempo=2.41min
🔍 Procesando: MaxAbs_FE_PCA30
✅ MaxAbs_FE_PCA30 [NONE]: F1_clase_0_test=0.174, Recall_clase_0_test=0.157, Tiempo=6.69min
🔍 Procesando: MaxAbs_FE_PCAopt
✅ MaxAbs_FE_PCA30 [NONE]: F1_clase_0_test=0.174, Recall_

KeyboardInterrupt: 